****

**IMPORTS**

In [ ]:
import os
import numpy as np
import torch
import torchvision.models as models
import torchvision.transforms as transforms
from PIL import Image
import matplotlib.pyplot as plt

**DATA PATH**

In [ ]:
root_dir = "/kaggle/input/datasets/jalalibrahim/tnbc-dataset/patches"

**LOAD IMAGE PATHS**

In [ ]:
image_paths = []

for patient in os.listdir(root_dir):
    patient_path = os.path.join(root_dir, patient)
    img_folder = os.path.join(patient_path, "images")

    if not os.path.exists(img_folder):
        continue

    for img in os.listdir(img_folder):
        if img.endswith(".png"):
            image_paths.append(os.path.join(img_folder, img))

print("Total patches:", len(image_paths))

**Model RESNET**

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

model = models.resnet50(weights=None)   # IMPORTANT: no internet
model = torch.nn.Sequential(*list(model.children())[:-1])

model = model.to(device)
model.eval()

**TRANSFORM**

In [ ]:
transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor()
])

**EMBEDDING FUNCTION**

In [ ]:
def get_embedding(path):
    img = Image.open(path).convert("RGB")
    img = transform(img).unsqueeze(0).to(device)

    with torch.no_grad():
        emb = model(img)

    return emb.cpu().numpy().flatten()

**EXTRACT EMBEDDINGS**

In [ ]:
embeddings = []
valid_paths = []

for i, p in enumerate(image_paths):
    try:
        emb = get_embedding(p)
        embeddings.append(emb)
        valid_paths.append(p)

        if i % 500 == 0:
            print("Processed:", i)

    except:
        pass

embeddings = np.array(embeddings)
print("Shape:", embeddings.shape)

**PCA (FAST CLUSTERING)**

In [ ]:
from sklearn.decomposition import PCA

pca = PCA(n_components=50)
embeddings_pca = pca.fit_transform(embeddings)
print("PCA shape:", embeddings_pca.shape)

**CLUSTERING**

In [ ]:
import hdbscan

clusterer = hdbscan.HDBSCAN(min_cluster_size=30)
clusters = clusterer.fit_predict(embeddings_pca)

print("Clusters:", len(set(clusters)))

In [ ]:
import numpy as np

unique, counts = np.unique(clusters, return_counts=True)
for u, c in zip(unique, counts):
    print("Cluster", u, ":", c)

**CLUSTER SIZE BAR CHART**

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

# Count clusters
cluster_counts = pd.Series(clusters).value_counts().sort_index()

plt.style.use("seaborn-v0_8-whitegrid")

fig, ax = plt.subplots(figsize=(9,6))

bars = ax.bar(
    cluster_counts.index.astype(str),
    cluster_counts.values,
    color="#4C72B0"
)

# highlight noise cluster (-1)
for i, c in enumerate(cluster_counts.index):
    if c == -1:
        bars[i].set_color("#D62728")

ax.set_title("Figure 2: Cluster Size Distribution in TNBC Embedding Space",
             fontsize=12, fontweight="bold")

ax.set_xlabel("Cluster ID")
ax.set_ylabel("Number of Patches")

ax.bar_label(bars, padding=3, fontsize=9)

plt.savefig("/kaggle/working/fig_cluster_distribution_ieee.png",
            dpi=600, bbox_inches="tight")

plt.show()

In [ ]:
plt.savefig(
    "/kaggle/working/fig_cluster_distribution.png",
    dpi=600,
    bbox_inches="tight"
)

**UMAP**

In [ ]:
import numpy as np
import umap

print("PCA shape:", embeddings_pca.shape)

reducer = umap.UMAP(
    n_neighbors=15,
    min_dist=0.1,
    random_state=42
)

umap_emb = reducer.fit_transform(embeddings_pca)

print("UMAP shape:", umap_emb.shape)

np.save("/kaggle/working/umap.npy", umap_emb)
print("Saved!")

**OUTLIERS (FAILURES)**

In [ ]:
from sklearn.ensemble import IsolationForest

clf = IsolationForest(contamination=0.05, random_state=42)
labels = clf.fit_predict(embeddings_pca)

outliers = np.where(labels == -1)[0]

print("Failure patches:", len(outliers))

**Paper PLOT**

**RAW UMAP**

In [ ]:
import matplotlib.pyplot as plt

plt.style.use("seaborn-v0_8-whitegrid")

fig, ax = plt.subplots(figsize=(9,7))

ax.scatter(
    umap_emb[:,0],
    umap_emb[:,1],
    c="#4D4D4D",   # professional dark gray
    s=2,
    alpha=0.5,
    rasterized=True
)

ax.set_title("Figure 1: Raw TNBC Patch Embedding Space (UMAP)",
             fontsize=13, fontweight="bold")

ax.set_xlabel("UMAP Dimension 1", fontsize=11)
ax.set_ylabel("UMAP Dimension 2", fontsize=11)

plt.savefig("/kaggle/working/fig1_ieee.png", dpi=600, bbox_inches="tight")
plt.show()

**CLUSTERS**

In [ ]:
import matplotlib.pyplot as plt

plt.style.use("seaborn-v0_8-whitegrid")

fig, ax = plt.subplots(figsize=(9,7))

ax.scatter(
    umap_emb[:,0],
    umap_emb[:,1],
    c=clusters,
    cmap="tab10",   # colorblind safe palette
    s=3,
    alpha=0.85
)

ax.set_title("Figure 2: TNBC Patch Clustering (HDBSCAN)",
             fontsize=13, fontweight="bold")

ax.set_xlabel("UMAP Dimension 1", fontsize=11)
ax.set_ylabel("UMAP Dimension 2", fontsize=11)

plt.savefig("/kaggle/working/fig2_ieee.png", dpi=600, bbox_inches="tight")
plt.show()

**FAILURE OVERLAY**

In [ ]:
import matplotlib.pyplot as plt

plt.style.use("seaborn-v0_8-whitegrid")

fig, ax = plt.subplots(figsize=(9,7))

# normal patches
ax.scatter(
    umap_emb[:,0],
    umap_emb[:,1],
    c="#BFBFBF",
    s=2,
    alpha=0.5,
    label="Normal patches"
)

# failure patches
ax.scatter(
    umap_emb[outliers,0],
    umap_emb[outliers,1],
    c="#D62728",
    s=10,
    edgecolors="black",
    linewidths=0.2,
    label="Failure-prone patches (405)"
)

ax.set_title("Figure 3: Failure-Prone Regions in TNBC Embedding Space",
             fontsize=13, fontweight="bold")

ax.set_xlabel("UMAP Dimension 1", fontsize=11)
ax.set_ylabel("UMAP Dimension 2", fontsize=11)

ax.legend(frameon=True)

plt.savefig("/kaggle/working/fig3_ieee.png", dpi=600, bbox_inches="tight")
plt.show()

In [ ]:
import numpy as np

unique, counts = np.unique(clusters, return_counts=True)
for u, c in zip(unique, counts):
    print("Cluster", u, ":", c)

In [ ]:
import pandas as pd

df = pd.DataFrame({
    "path": valid_paths,
    "cluster": clusters,
    "failure": np.isin(range(len(valid_paths)), outliers)
})

df.to_csv("/kaggle/working/results.csv", index=False)

In [ ]:
import os
import numpy as np
import pandas as pd

# 🔹 Create output folder
save_dir = "/kaggle/working/tnbc_results"
os.makedirs(save_dir, exist_ok=True)

# 🔹 Save embeddings
np.save(os.path.join(save_dir, "embeddings.npy"), embeddings_pca)

# 🔹 Save UMAP
np.save(os.path.join(save_dir, "umap.npy"), umap_emb)

# 🔹 Save clusters
np.save(os.path.join(save_dir, "clusters.npy"), clusters)

# 🔹 Save outliers (failures)
np.save(os.path.join(save_dir, "outliers.npy"), outliers)

# 🔹 Save metadata table
df = pd.DataFrame({
    "path": valid_paths,
    "cluster": clusters,
    "failure": np.isin(range(len(valid_paths)), outliers)
})

df.to_csv(os.path.join(save_dir, "TNBCresults.csv"), index=False)

print("✅ All results saved at:", save_dir)

In [ ]:
import os

save_dir = "/kaggle/working/tnbc_figures"
os.makedirs(save_dir, exist_ok=True)

In [ ]:
plt.savefig(
    save_dir + "/fig1_raw_umap.png",
    dpi=600,
    bbox_inches="tight"
)

In [ ]:
plt.savefig(
    save_dir + "/fig2_clusters.png",
    dpi=600,
    bbox_inches="tight"
)

In [ ]:
plt.savefig(
    save_dir + "/fig3_failures.png",
    dpi=600,
    bbox_inches="tight"
)